# FireSight-IR | Module 1c v2 — Visualization & LinkedIn Figures

**Purpose:** Generate publication-quality figures from Module 1c v2 outputs for LinkedIn and reporting.  
**Depends on:** Module 1c v2 parquets in `data/processed/viirs_fp_srf_v2/`

## Figures produced
1. **Spatial fire pixel map** — all 1.15M pixels across western CONUS, coloured by land cover
2. **Land cover breakdown** — MODIS land cover distribution across fire pixels
3. **Label distribution comparison** — false-alarm locations overlaid on land cover
4. **Feature completeness audit** — before (v1 zeros) vs after (v2 real data)
5. **BTD separation** — wildfire vs false-alarm brightness temperature difference
6. **LinkedIn hero figure** — composite 2-panel designed for posting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install pandas numpy matplotlib cartopy pyarrow scipy h5py -q

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
from matplotlib.ticker import FuncFormatter
import matplotlib.patheffects as pe
import h5py
warnings.filterwarnings('ignore')

# Try importing cartopy for map backgrounds
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
    print('Cartopy available.')
except ImportError:
    HAS_CARTOPY = False
    print('Cartopy not available — using plain axes for maps.')

print('All imports OK.')

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
SRF_V2_DIR  = f'{BASE_DIR}/data/processed/viirs_fp_srf_v2'
FIGURES_DIR = f'{BASE_DIR}/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

ALL_YEARS = [2018, 2019, 2020, 2021, 2022, 2023]

# ── Design system ─────────────────────────────────────────────────────────────
BG      = '#0D1117'    # near-black background
PANEL   = '#161B22'    # slightly lighter panel
BORDER  = '#30363D'
TEXT    = '#E6EDF3'
SUBTEXT = '#8B949E'
ORANGE  = '#E85D04'
NAVY    = '#1565C0'
TEAL    = '#0D9488'
AMBER   = '#D97706'
GREEN   = '#16A34A'
RED     = '#DC2626'

# Land cover colour palette
LC_COLORS = {
    'forest':    '#1A6B2F',
    'grassland': '#78A830',
    'shrub':     '#C8963C',
    'cropland':  '#E8C547',
    'urban':     '#E85D04',
    'bare':      '#8B7355',
    'water':     '#2563EB',
    'unknown':   '#444444',
}

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   PANEL,
    'axes.edgecolor':   BORDER,
    'axes.labelcolor':  TEXT,
    'xtick.color':      SUBTEXT,
    'ytick.color':      SUBTEXT,
    'text.color':       TEXT,
    'grid.color':       BORDER,
    'grid.linewidth':   0.5,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

print('Configuration ready.')

---
## Section 1 — Load all v2 parquets

In [ ]:
dfs = []
for year in ALL_YEARS:
    fp = f'{SRF_V2_DIR}/viirs_fp_srf_v2_{year}.parquet'
    if os.path.exists(fp):
        df = pd.read_parquet(fp)
        df['year'] = year
        dfs.append(df)
        print(f'  {year}: {len(df):>9,} pixels | {len(df.columns)} cols')
    else:
        print(f'  {year}: NOT FOUND')

assert dfs, 'No v2 parquets found. Run Module 1c v2 first.'
df_all = pd.concat(dfs, ignore_index=True)

# Compute derived columns for plotting
lc_cols = ['lc_forest','lc_shrub','lc_grassland','lc_cropland','lc_urban','lc_bare','lc_water']
if all(c in df_all.columns for c in lc_cols):
    lc_arr = df_all[lc_cols].values
    lc_sums = lc_arr.sum(axis=1)
    lc_names = ['forest','shrub','grassland','cropland','urban','bare','water']
    df_all['lc_dominant'] = np.where(
        lc_sums == 0, 'unknown',
        pd.DataFrame(lc_arr, columns=lc_names).idxmax(axis=1)
    )
else:
    df_all['lc_dominant'] = 'unknown'

# BTD
if 'BT_I4' in df_all.columns and 'BT_I5' in df_all.columns:
    df_all['BTD'] = df_all['BT_I4'] - df_all['BT_I5']
elif 'BTD' not in df_all.columns:
    df_all['BTD'] = np.nan

print(f'\nTotal: {len(df_all):,} pixels')
print(f'Labels: {dict(df_all["label"].value_counts().sort_index())}')
print(f'LC dominant: {dict(df_all["lc_dominant"].value_counts().head(5))}')

---
## Section 2 — Spatial fire pixel map

In [ ]:
# Sample for speed (full 1.15M is too dense to render clearly)
SAMPLE_N = 150_000
rng = np.random.default_rng(42)
idx = rng.choice(len(df_all), size=min(SAMPLE_N, len(df_all)), replace=False)
df_sample = df_all.iloc[idx].copy()

lons = df_sample['longitude'].values
lats = df_sample['latitude'].values
lc   = df_sample['lc_dominant'].values

# Map colours per pixel
colors = np.array([LC_COLORS.get(l, LC_COLORS['unknown']) for l in lc])

if HAS_CARTOPY:
    fig = plt.figure(figsize=(14, 9), facecolor=BG)
    ax  = fig.add_subplot(1, 1, 1, projection=ccrs.AlbersEqualArea(
        central_longitude=-117, central_latitude=40.5))
    ax.set_facecolor(BG)
    ax.set_extent([-125, -109, 32, 49], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN,       facecolor='#0D1117', zorder=0)
    ax.add_feature(cfeature.LAND,        facecolor='#1A1A2E', zorder=0)
    ax.add_feature(cfeature.STATES,      edgecolor=BORDER, linewidth=0.6, zorder=2)
    ax.add_feature(cfeature.COASTLINE,   edgecolor=SUBTEXT, linewidth=0.5, zorder=2)
    ax.add_feature(cfeature.BORDERS,     edgecolor=SUBTEXT, linewidth=0.7, zorder=2)

    # Sort by lc so rarer classes appear on top
    order = np.argsort([{'forest':0,'grassland':1,'shrub':2,'cropland':3,
                          'bare':4,'water':5,'urban':6,'unknown':7}.get(l,9) for l in lc])
    for lc_type, color in LC_COLORS.items():
        mask = lc == lc_type
        if mask.sum() == 0: continue
        size  = 6 if lc_type == 'urban' else 1.5
        alpha = 0.9 if lc_type == 'urban' else 0.35
        ax.scatter(lons[mask], lats[mask], s=size, c=color, alpha=alpha,
                   transform=ccrs.PlateCarree(), zorder=3, linewidths=0,
                   label=lc_type)
else:
    fig, ax = plt.subplots(1, 1, figsize=(14, 9), facecolor=BG)
    ax.set_facecolor('#1A1A2E')
    for lc_type, color in LC_COLORS.items():
        mask = lc == lc_type
        if mask.sum() == 0: continue
        size  = 8 if lc_type == 'urban' else 1.5
        alpha = 0.9 if lc_type == 'urban' else 0.35
        ax.scatter(lons[mask], lats[mask], s=size, c=color, alpha=alpha,
                   linewidths=0, label=lc_type)
    ax.set_xlim(-125.5, -108.5)
    ax.set_ylim(31.5, 49.5)
    ax.set_xlabel('Longitude', fontsize=10, color=SUBTEXT)
    ax.set_ylabel('Latitude',  fontsize=10, color=SUBTEXT)

# Legend
legend_handles = [
    mpatches.Patch(color=LC_COLORS[k], label=k.capitalize())
    for k in ['forest','grassland','shrub','cropland','urban','bare','water']
]
leg = ax.legend(handles=legend_handles, loc='lower left',
                facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT,
                fontsize=9, ncol=2, framealpha=0.9,
                title='Land cover', title_fontsize=9)
leg.get_title().set_color(SUBTEXT)

# Title and annotations
fig.text(0.5, 0.96, 'FireSight-IR  |  1.15 Million VIIRS Fire Pixels  |  Western CONUS  |  2018–2023',
         ha='center', va='top', fontsize=13, color=TEXT, fontweight='bold')
fig.text(0.5, 0.92, 'Coloured by MODIS MCD12Q1 v6.1 land cover  ·  Urban pixels (orange) enlarged for visibility',
         ha='center', va='top', fontsize=10, color=SUBTEXT)

# Stats annotations
stats_text = f'Total pixels: {len(df_all):,}\nFire seasons: May–Oct\nSample shown: {len(df_sample):,}'
fig.text(0.82, 0.15, stats_text, fontsize=9, color=SUBTEXT,
         bbox=dict(boxstyle='round,pad=0.4', facecolor=PANEL, edgecolor=BORDER, alpha=0.8))

plt.tight_layout(rect=[0, 0, 1, 0.91])
out = f'{FIGURES_DIR}/1c_spatial_map.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section 3 — LinkedIn hero figure (composite)

In [ ]:
fig = plt.figure(figsize=(16, 9), facecolor=BG)
fig.patch.set_facecolor(BG)

# 3-panel layout: left map, top-right land cover bar, bottom-right BTD violin
gs = gridspec.GridSpec(2, 2, figure=fig,
                       left=0.04, right=0.97,
                       top=0.88, bottom=0.08,
                       wspace=0.28, hspace=0.42)

# ── Panel A: spatial map ─────────────────────────────────────────────────────
if HAS_CARTOPY:
    ax_map = fig.add_subplot(gs[:, 0], projection=ccrs.AlbersEqualArea(
        central_longitude=-117, central_latitude=40.5))
    ax_map.set_facecolor('#0D1520')
    ax_map.set_extent([-125, -109, 32, 49], crs=ccrs.PlateCarree())
    ax_map.add_feature(cfeature.OCEAN,    facecolor='#0A1220', zorder=0)
    ax_map.add_feature(cfeature.LAND,     facecolor='#0D1520', zorder=0)
    ax_map.add_feature(cfeature.STATES,   edgecolor='#2D3748', linewidth=0.7, zorder=2)
    ax_map.add_feature(cfeature.COASTLINE,edgecolor='#4A5568', linewidth=0.5, zorder=2)
    for lc_type, color in LC_COLORS.items():
        mask = lc == lc_type
        if mask.sum() == 0: continue
        s = 8 if lc_type == 'urban' else 1.2
        a = 0.95 if lc_type == 'urban' else 0.3
        ax_map.scatter(lons[mask], lats[mask], s=s, c=color, alpha=a,
                       transform=ccrs.PlateCarree(), zorder=3, linewidths=0)
else:
    ax_map = fig.add_subplot(gs[:, 0])
    ax_map.set_facecolor('#0D1520')
    for lc_type, color in LC_COLORS.items():
        mask = lc == lc_type
        if mask.sum() == 0: continue
        ax_map.scatter(lons[mask], lats[mask], s=1.5 if lc_type!='urban' else 8,
                       c=color, alpha=0.3 if lc_type!='urban' else 0.9, linewidths=0)
    ax_map.set_xlim(-125.5, -108.5); ax_map.set_ylim(31.5, 49.5)

ax_map.set_title('1.15M VIIRS fire pixels  ·  Western CONUS  ·  2018–2023',
                 color=TEXT, fontsize=10, pad=8)

# Panel A legend
leg_h = [mpatches.Patch(color=LC_COLORS[k], label=k.capitalize())
         for k in ['forest','grassland','shrub','cropland','urban','bare']]
leg = ax_map.legend(handles=leg_h, loc='lower left', facecolor=PANEL,
                    edgecolor=BORDER, labelcolor=TEXT, fontsize=8,
                    ncol=2, framealpha=0.85, title='Land cover', title_fontsize=8)
leg.get_title().set_color(SUBTEXT)

# ── Panel B: land cover horizontal bar chart ──────────────────────────────────
ax_lc = fig.add_subplot(gs[0, 1])
ax_lc.set_facecolor(PANEL)

lc_counts = df_all['lc_dominant'].value_counts()
total_px   = len(df_all)
lc_order   = ['grassland','forest','shrub','cropland','urban','bare','water','unknown']
lc_order   = [l for l in lc_order if l in lc_counts.index]
lc_vals    = [lc_counts.get(l, 0) / total_px * 100 for l in lc_order]
lc_bar_cols= [LC_COLORS.get(l, '#444444') for l in lc_order]

bars = ax_lc.barh(range(len(lc_order)), lc_vals, color=lc_bar_cols, height=0.7)
ax_lc.set_yticks(range(len(lc_order)))
ax_lc.set_yticklabels([l.capitalize() for l in lc_order], fontsize=10, color=TEXT)
ax_lc.set_xlabel('% of fire pixels', fontsize=9, color=SUBTEXT)
ax_lc.set_xlim(0, max(lc_vals) * 1.18)
ax_lc.invert_yaxis()
ax_lc.grid(axis='x', alpha=0.3)
ax_lc.spines[['top','right','left']].set_visible(False)

for bar, val in zip(bars, lc_vals):
    ax_lc.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
               f'{val:.1f}%', va='center', fontsize=9, color=TEXT, fontweight='bold')

ax_lc.set_title('Land cover distribution', color=TEXT, fontsize=11, pad=6)

# Add annotation: "MODIS MCD12Q1 v6.1"
ax_lc.text(0.98, 0.02, 'Source: MODIS MCD12Q1 v6.1', transform=ax_lc.transAxes,
           ha='right', va='bottom', fontsize=8, color=SUBTEXT)

# ── Panel C: BTD distribution by label ───────────────────────────────────────
ax_btd = fig.add_subplot(gs[1, 1])
ax_btd.set_facecolor(PANEL)

if 'BTD' in df_all.columns and not df_all['BTD'].isna().all():
    # Sample for violin
    for label_val, name, color in [
        (1, 'Wildfire',     ORANGE),
        (2, 'False-alarm',  NAVY),
        (0, 'Non-fire',     TEAL),
    ]:
        sub = df_all[df_all['label'] == label_val]['BTD'].dropna()
        if len(sub) == 0: continue
        sub_sample = sub.sample(min(30000, len(sub)), random_state=42)
        # KDE-based density estimate
        from scipy import stats
        x_range = np.linspace(sub_sample.quantile(0.01), sub_sample.quantile(0.99), 300)
        kde  = stats.gaussian_kde(sub_sample, bw_method=0.3)
        dens = kde(x_range)
        dens = dens / dens.max()
        ax_btd.fill_between(x_range, dens, alpha=0.35, color=color)
        ax_btd.plot(x_range, dens, color=color, linewidth=1.8, label=f'{name} (n={len(sub):,})')
        # Median line
        med = sub_sample.median()
        ax_btd.axvline(med, color=color, linewidth=1, linestyle='--', alpha=0.6)

    # Physics threshold line
    ax_btd.axvline(10, color='white', linewidth=1.2, linestyle=':', alpha=0.5)
    ax_btd.text(10.5, 0.95, 'BTD=10K\nphysics threshold', va='top', fontsize=8,
               color=SUBTEXT, transform=ax_btd.get_xaxis_transform())

    ax_btd.set_xlabel('BTD = BT_I4 − BT_I5 (K)', fontsize=9, color=SUBTEXT)
    ax_btd.set_ylabel('Normalised density', fontsize=9, color=SUBTEXT)
    ax_btd.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
    ax_btd.spines[['top','right']].set_visible(False)
    ax_btd.set_title('BTD separation: wildfire vs false-alarm', color=TEXT, fontsize=11, pad=6)
    ax_btd.grid(alpha=0.2)
else:
    ax_btd.text(0.5, 0.5, 'BTD data not available\n(BT_I4/BT_I5 not in parquet)',
                ha='center', va='center', color=SUBTEXT, fontsize=11,
                transform=ax_btd.transAxes)
    ax_btd.set_title('BTD separation', color=TEXT, fontsize=11, pad=6)

# ── Header ───────────────────────────────────────────────────────────────────
fig.text(0.5, 0.97,
         'FireSight-IR  |  Module 1c v2  —  Surface Context & Label Construction',
         ha='center', va='top', fontsize=15, fontweight='bold', color=TEXT)
fig.text(0.5, 0.93,
         'MODIS MCD12Q1 land cover · OSM infrastructure · 1,149,722 VIIRS fire pixels · Western CONUS 2018–2023',
         ha='center', va='top', fontsize=10, color=SUBTEXT)

# ── Footer ───────────────────────────────────────────────────────────────────
fig.text(0.5, 0.02,
         'FireSight-IR  ·  github.com/Ibekwemmanuel7  ·  FireSat Protoflight-aligned wildfire detection  ·  VIIRS + ERA5 + MODIS',
         ha='center', va='bottom', fontsize=8, color=SUBTEXT)

out_linkedin = f'{FIGURES_DIR}/1c_linkedin_hero.png'
plt.savefig(out_linkedin, dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'\n✓ LinkedIn hero figure saved → {out_linkedin}')

---
## Section 4 — Feature completeness comparison (v1 zeros vs v2 real)

In [ ]:
features_v1 = {
    'ERA5 atmospheric (16)':  {'v1': 100, 'v2': 100, 'type': 'atm'},
    'MODIS land cover (8)':   {'v1': 0,   'v2': 100, 'type': 'srf'},
    'OSM distances (5)':      {'v1': 87,  'v2': 87,  'type': 'srf'},
    'Solar geometry (2)':     {'v1': 100, 'v2': 100, 'type': 'srf'},
    'DNB emitters (2)':       {'v1': 0,   'v2': 0,   'type': 'srf'},
    'Burn scars (2)':         {'v1': 0,   'v2': 0,   'type': 'srf'},
    'Derived physics (6)':    {'v1': 100, 'v2': 100, 'type': 'derived'},
}

labels_feat = list(features_v1.keys())
v1_vals = [features_v1[k]['v1'] for k in labels_feat]
v2_vals = [features_v1[k]['v2'] for k in labels_feat]

fig, ax = plt.subplots(figsize=(12, 5.5), facecolor=BG)
ax.set_facecolor(PANEL)

x = np.arange(len(labels_feat))
w = 0.32
bars1 = ax.bar(x - w/2, v1_vals, w, label='v1 (ESA CCI — failed)', color='#DC262660', 
               edgecolor='#DC2626', linewidth=1.2)
bars2 = ax.bar(x + w/2, v2_vals, w, label='v2 (MODIS MCD12Q1)', color='#16A34A60',
               edgecolor='#16A34A', linewidth=1.2)

# Annotate improvement
for i, (v1, v2) in enumerate(zip(v1_vals, v2_vals)):
    if v2 > v1:
        ax.annotate('', xy=(x[i]+w/2, v2+2), xytext=(x[i]-w/2, v1+2),
                    arrowprops=dict(arrowstyle='->', color=GREEN, lw=1.5))
        ax.text(x[i], max(v1,v2)+6, f'+{v2-v1}pp', ha='center', fontsize=9,
                color=GREEN, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels_feat, rotation=18, ha='right', fontsize=10, color=TEXT)
ax.set_ylabel('Feature coverage (%)', fontsize=10, color=SUBTEXT)
ax.set_ylim(0, 118)
ax.set_yticks([0, 25, 50, 75, 100])
ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
ax.set_title('Feature coverage: Module 1c v1 (ESA CCI) vs v2 (MODIS MCD12Q1)',
             color=TEXT, fontsize=12, pad=10)

# Annotation box
ax.text(0.99, 0.97,
        'ESA CCI failed: CDS API 401 auth error\nReplaced with MODIS via NASA earthaccess',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color=SUBTEXT,
        bbox=dict(boxstyle='round,pad=0.4', facecolor=BG, edgecolor=BORDER, alpha=0.85))

plt.tight_layout()
out = f'{FIGURES_DIR}/1c_feature_comparison.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section 5 — False-alarm spatial overlay

In [ ]:
df_wf = df_all[df_all['label'] == 1].sample(min(80000, (df_all['label']==1).sum()), random_state=42)
df_fa = df_all[df_all['label'] == 2]

print(f'Wildfire sample: {len(df_wf):,}')
print(f'False-alarm    : {len(df_fa):,}')

if HAS_CARTOPY:
    fig = plt.figure(figsize=(13, 8.5), facecolor=BG)
    ax  = fig.add_subplot(1,1,1, projection=ccrs.AlbersEqualArea(
        central_longitude=-117, central_latitude=40.5))
    ax.set_facecolor('#0A1220')
    ax.set_extent([-125, -109, 32, 49], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.OCEAN,    facecolor='#0A1220', zorder=0)
    ax.add_feature(cfeature.LAND,     facecolor='#0D1520', zorder=0)
    ax.add_feature(cfeature.STATES,   edgecolor='#2D3748', linewidth=0.7, zorder=2)
    ax.add_feature(cfeature.COASTLINE,edgecolor='#4A5568', linewidth=0.5, zorder=2)
    ax.scatter(df_wf['longitude'], df_wf['latitude'], s=0.8, c=ORANGE, alpha=0.2,
               transform=ccrs.PlateCarree(), zorder=3, linewidths=0, label='Wildfire')
    ax.scatter(df_fa['longitude'], df_fa['latitude'], s=12, c='#3B82F6', alpha=0.85,
               transform=ccrs.PlateCarree(), zorder=5, linewidths=0, label=f'False-alarm (n={len(df_fa):,})')
else:
    fig, ax = plt.subplots(1,1, figsize=(13, 8.5), facecolor=BG)
    ax.set_facecolor('#0D1520')
    ax.scatter(df_wf['longitude'], df_wf['latitude'], s=0.8, c=ORANGE, alpha=0.2, linewidths=0, label='Wildfire')
    ax.scatter(df_fa['longitude'], df_fa['latitude'], s=12,  c='#3B82F6', alpha=0.85, linewidths=0, label=f'False-alarm (n={len(df_fa):,})')
    ax.set_xlim(-125.5, -108.5); ax.set_ylim(31.5, 49.5)

ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=10,
          markerscale=4, loc='lower left')
fig.text(0.5, 0.96, 'FireSight-IR  |  False-alarm Locations vs Wildfire Pixels',
         ha='center', va='top', fontsize=13, color=TEXT, fontweight='bold')
fig.text(0.5, 0.92, 'False-alarms (blue) detected via MODIS urban land cover + OSM industrial proximity gate',
         ha='center', va='top', fontsize=10, color=SUBTEXT)

plt.tight_layout(rect=[0,0,1,0.91])
out = f'{FIGURES_DIR}/1c_false_alarm_map.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section 6 — Annual fire pixel counts (timeline)

In [ ]:
year_counts = df_all.groupby(['year','label']).size().unstack(fill_value=0)
years_plot  = sorted(df_all['year'].unique())

wf_counts = [year_counts.get(1, pd.Series()).get(y, 0) for y in years_plot]
fa_counts = [year_counts.get(2, pd.Series()).get(y, 0) for y in years_plot]
nf_counts = [year_counts.get(0, pd.Series()).get(y, 0) for y in years_plot]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG)
fig.patch.set_facecolor(BG)

# Left: stacked bar
ax = axes[0]
ax.set_facecolor(PANEL)
x = np.arange(len(years_plot))
ax.bar(x, wf_counts, color=ORANGE, alpha=0.85, label='Wildfire')
ax.bar(x, fa_counts, bottom=wf_counts, color=NAVY, alpha=0.85, label='False-alarm')
ax.bar(x, nf_counts, bottom=[w+f for w,f in zip(wf_counts,fa_counts)],
       color=TEAL, alpha=0.85, label='Non-fire')
ax.set_xticks(x); ax.set_xticklabels(years_plot, color=TEXT)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v/1e3:.0f}K'))
ax.set_ylabel('Fire pixels', color=SUBTEXT)
ax.set_title('Annual fire pixel counts by label', color=TEXT, fontsize=11)
ax.legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)

# Annotate 2020 peak
ax.annotate('2020 mega-fire\nyear', xy=(2, wf_counts[2]), xytext=(3.4, wf_counts[2]*1.05),
            fontsize=8, color=SUBTEXT,
            arrowprops=dict(arrowstyle='->', color=SUBTEXT, lw=1))

# Right: false-alarm % by year
ax2 = axes[1]
ax2.set_facecolor(PANEL)
total_by_year = [w+f+n for w,f,n in zip(wf_counts,fa_counts,nf_counts)]
fa_pct = [100*f/t if t>0 else 0 for f,t in zip(fa_counts,total_by_year)]
bars = ax2.bar(x, fa_pct, color=[NAVY if p > 4 else '#3A5C8A' for p in fa_pct], alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(years_plot, color=TEXT)
ax2.set_ylabel('False-alarm %', color=SUBTEXT)
ax2.set_title('False-alarm rate by year\n(MODIS urban + OSM industrial detection)', color=TEXT, fontsize=11)
ax2.spines[['top','right']].set_visible(False)
ax2.grid(axis='y', alpha=0.2)
for bar, val in zip(bars, fa_pct):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05, f'{val:.1f}%',
             ha='center', fontsize=9, color=TEXT, fontweight='bold')

ax2.text(0.98, 0.97, '2023 = held-out validation year', transform=ax2.transAxes,
         ha='right', va='top', fontsize=8, color=SUBTEXT)

fig.suptitle('FireSight-IR  |  Dataset Overview by Year', color=TEXT, fontsize=13, y=1.01)
plt.tight_layout()
out = f'{FIGURES_DIR}/1c_annual_counts.png'
plt.savefig(out, dpi=180, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out}')

---
## Section 7 — Figure inventory

In [ ]:
import os
print('=== Module 1c v2 Figures ===')
figs = [
    ('1c_linkedin_hero.png',      'LinkedIn hero composite — BEST for posting'),
    ('1c_spatial_map.png',        'Full spatial map of fire pixels by land cover'),
    ('1c_false_alarm_map.png',    'False-alarm locations vs wildfire pixels'),
    ('1c_feature_comparison.png', 'Feature coverage v1 vs v2'),
    ('1c_annual_counts.png',      'Annual pixel counts and false-alarm rate'),
]
for fname, desc in figs:
    fpath = f'{FIGURES_DIR}/{fname}'
    exists = '✓' if os.path.exists(fpath) else '✗'
    size = f'{os.path.getsize(fpath)/1e6:.1f} MB' if os.path.exists(fpath) else 'missing'
    print(f'  {exists}  {fname:<40} {size:>8}   {desc}')
print(f'\n  All figures saved to: {FIGURES_DIR}')
print(f'  Recommended for LinkedIn: 1c_linkedin_hero.png')